# Transformers from Scratch

Notebook for **Chapter 11** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch11.html

Tip: change a number, predict the output, then run the cell again.

## 11.1 Self-Attention by Hand

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)       # subtract max: avoids overflow
    e = np.exp(z); return e / e.sum(axis=axis, keepdims=True)
X  = np.array([[1.,0.],[0.,1.],[1.,1.]])
Wq = np.array([[1.,0.],[0.,1.]])
Wk = np.array([[1.,1.],[0.,1.]])
Wv = np.array([[2.,0.],[0.,3.]])
Q, K, V = X@Wq, X@Wk, X@Wv
S = Q @ K.T / np.sqrt(2)
A = softmax(S)
print("scaled scores\n", S); print("weights\n", A, A.sum(1)); print("output\n", A @ V)

scaled scores
 [[0.7071 0.     0.7071]
 [0.7071 0.7071 1.4142]
 [1.4142 0.7071 2.1213]]
weights
 [[0.4011 0.1978 0.4011]
 [0.2483 0.2483 0.5035]
 [0.284  0.14   0.576 ]] [1. 1. 1.]
output
 [[1.6044 1.7967]
 [1.5035 2.2552]
 [1.7199 2.148  ]]


In [ ]:
rng = np.random.default_rng(0)
for d in (4, 64, 512):
    q = rng.standard_normal((20000, d)); k = rng.standard_normal((20000, d))
    dots = (q * k).sum(1)
    print(d, "std of dot", dots.std().round(2), " scaled", (dots / np.sqrt(d)).std().round(2))
z = np.array([1., 2., 3.]) * 8
print(softmax(z), softmax(z / 8))

4 std of dot 2.0  scaled 1.0
64 std of dot 8.03  scaled 1.0
512 std of dot 22.54  scaled 1.0
[0.     0.0003 0.9997] [0.09   0.2447 0.6652]


## 11.2 Causal Masking and Multi-Head Attention

In [ ]:
mask = np.triu(np.ones((3, 3), bool), 1)         # True above the diagonal
Sm = np.where(mask, -np.inf, S)
print(softmax(Sm)); print(softmax(Sm) @ V)

[[1.    0.    0.   ]
 [0.5   0.5   0.   ]
 [0.284 0.14  0.576]]
[[2.    0.    ]
 [1.    1.5   ]
 [1.7199 2.148 ]]


In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True); e = np.exp(z); return e/e.sum(axis=axis, keepdims=True)
rng = np.random.default_rng(0)
B, T, D, H = 2, 5, 8, 2
x = rng.standard_normal((B, T, D))
Wq, Wk, Wv, Wo = [rng.standard_normal((D, D)) * 0.3 for _ in range(4)]
def split_heads(t):                       # (B,T,D) -> (B,H,T,dh)
    return t.reshape(B, T, H, D // H).transpose(0, 2, 1, 3)
q, k, v = split_heads(x @ Wq), split_heads(x @ Wk), split_heads(x @ Wv)
s = q @ k.transpose(0, 1, 3, 2) / np.sqrt(D // H)
print("scores", s.shape)
a = softmax(s)
print("attention", a.shape, "row sums", a.sum(-1).min(), a.sum(-1).max())
heads = a @ v
print("per-head output", heads.shape)
concat = heads.transpose(0, 2, 1, 3).reshape(B, T, D)
print("concatenated", concat.shape)
out = concat @ Wo
print("final", out.shape)
print("params of one attention layer, D=8:", 4*D*D, "; D=768:", 4*768*768)

scores (2, 2, 5, 5)
attention (2, 2, 5, 5) row sums 0.9999999999999998 1.0000000000000002
per-head output (2, 2, 5, 4)
concatenated (2, 5, 8)
final (2, 5, 8)
params of one attention layer, D=8: 256 ; D=768: 2359296


## 11.3 Positional Encodings: Telling the Model Word Order

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
def sinpe(T, d):
    pos = np.arange(T)[:, None]; i = np.arange(d // 2)[None, :]
    ang = pos / 10000 ** (2 * i / d)
    pe = np.zeros((T, d)); pe[:, 0::2] = np.sin(ang); pe[:, 1::2] = np.cos(ang)
    return pe
print(sinpe(4, 4))
pe = sinpe(50, 16)
print(round(pe[5] @ pe[8], 4), round(pe[20] @ pe[23], 4), round(pe[5] @ pe[20], 4))
def rot(v, pos, theta=1.0):
    c, s = np.cos(pos * theta), np.sin(pos * theta)
    return np.array([c * v[0] - s * v[1], s * v[0] + c * v[1]])
q = np.array([1., 2.]); k = np.array([0.5, -1.])
for m, n in [(3, 1), (10, 8), (100, 98)]:
    print("rope positions", m, n, "dot", round(rot(q, m) @ rot(k, n), 6))
print("plain dot", q @ k)
T = 4; slope = 0.5
dist = np.arange(T)[:, None] - np.arange(T)[None, :]
print(np.where(dist >= 0, -slope * dist, -np.inf))

[[ 0.      1.      0.      1.    ]
 [ 0.8415  0.5403  0.01    1.    ]
 [ 0.9093 -0.4161  0.02    0.9998]
 [ 0.1411 -0.99    0.03    0.9996]]
5.5431 5.5431 4.2192
rope positions 3 1 dot -1.194375
rope positions 10 8 dot -1.194375
rope positions 100 98 dot -1.194375
plain dot -1.5
[[-0.  -inf -inf -inf]
 [-0.5 -0.  -inf -inf]
 [-1.  -0.5 -0.  -inf]
 [-1.5 -1.  -0.5 -0. ]]


## 11.4 The Transformer Block and a Complete Tiny GPT

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True); e = np.exp(z); return e / e.sum(axis=axis, keepdims=True)
def layernorm(x, g, b, eps=1e-5):
    m = x.mean(-1, keepdims=True); v = x.var(-1, keepdims=True)
    return g * (x - m) / np.sqrt(v + eps) + b
gelu = lambda x: 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

print("LN of [1,2,3,6]:", layernorm(np.array([[1., 2., 3., 6.]]), 1, 0))

rng = np.random.default_rng(1)
V, T, D, H, F, L = 11, 6, 16, 4, 64, 2      # vocab, context, width, heads, FFN width, layers
def n(*shape): return rng.standard_normal(shape) * 0.1
P = {"tok": n(V, D), "pos": n(T, D), "gf": np.ones(D), "bf": np.zeros(D), "blocks": []}
for _ in range(L):
    P["blocks"].append(dict(Wqkv=n(D, 3*D), Wo=n(D, D), W1=n(D, F), b1=np.zeros(F), W2=n(F, D), b2=np.zeros(D),
                            g1=np.ones(D), be1=np.zeros(D), g2=np.ones(D), be2=np.zeros(D)))

def block(x, p):
    B, T, D = x.shape; dh = D // H
    h = layernorm(x, p["g1"], p["be1"])                       # pre-norm
    q, k, v = np.split(h @ p["Wqkv"], 3, -1)
    sp = lambda t: t.reshape(B, T, H, dh).transpose(0, 2, 1, 3)
    q, k, v = sp(q), sp(k), sp(v)
    s = q @ k.transpose(0, 1, 3, 2) / np.sqrt(dh)
    s = np.where(np.triu(np.ones((T, T), bool), 1), -np.inf, s)   # causal mask
    o = (softmax(s) @ v).transpose(0, 2, 1, 3).reshape(B, T, D)
    x = x + o @ p["Wo"]                                       # residual 1
    h = layernorm(x, p["g2"], p["be2"])
    return x + gelu(h @ p["W1"] + p["b1"]) @ p["W2"] + p["b2"]    # residual 2

def gpt(ids):
    x = P["tok"][ids] + P["pos"][:ids.shape[1]]
    for p in P["blocks"]: x = block(x, p)
    return layernorm(x, P["gf"], P["bf"]) @ P["tok"].T         # weight tying

ids = rng.integers(0, V, (3, T))
logits = gpt(ids); print("logits", logits.shape)
pr = softmax(logits); tgt = np.roll(ids, -1, 1)
loss = -np.log(np.take_along_axis(pr, tgt[..., None], -1)).mean()
print("loss", round(loss, 4), " ln(V) =", round(np.log(V), 4))
count = lambda o: sum(count(i) for i in (o.values() if isinstance(o, dict) else o)) if isinstance(o, (dict, list)) else o.size
print("parameters", count(P))
ids2 = ids.copy(); ids2[:, -1] = (ids2[:, -1] + 1) % V
print("earlier positions unchanged:", np.allclose(gpt(ids)[:, :-1], gpt(ids2)[:, :-1]))
print("last position changed:", not np.allclose(gpt(ids)[:, -1], gpt(ids2)[:, -1]))
seq = np.array([[3]])
for _ in range(5):
    seq = np.concatenate([seq, gpt(seq)[:, -1].argmax(-1)[:, None]], 1)
print("greedy generation from token 3:", seq)

LN of [1,2,3,6]: [[-1.069  -0.5345  0.      1.6036]]
logits (3, 6, 11)
loss 2.5271  ln(V) = 2.3979
parameters 6736
earlier positions unchanged: True
last position changed: True
greedy generation from token 3: [[3 0 0 0 9 9]]


In [ ]:
import numpy as np
def softmax(z): z = z - z.max(); e = np.exp(z); return e / e.sum()
rng = np.random.default_rng(0)
d = 4
Wq, Wk, Wv = [rng.standard_normal((d, d)) for _ in range(3)]
xs = rng.standard_normal((6, d))
def recompute_last(xs):
    q, k, v = xs @ Wq, xs @ Wk, xs @ Wv
    return softmax(q[-1] @ k.T / np.sqrt(d)) @ v
Kc, Vc = [], []
for t in range(6):                       # one new token per step
    Kc.append(xs[t] @ Wk); Vc.append(xs[t] @ Wv)     # only the NEW k and v are computed
    out = softmax((xs[t] @ Wq) @ np.array(Kc).T / np.sqrt(d)) @ np.array(Vc)
print("cache == recompute:", np.allclose(out, recompute_last(xs)))
print("K,V projections without cache:", sum(range(1, 7)), " with cache:", 6)
for T in (1024, 8192, 32768):
    print(T, "tokens: score matrix per head in fp16 =", round(T*T*2/1e6, 1), "MB")
kv = 2 * 32 * 32 * 128 * 4096 * 2
print("KV cache, 32 layers, 32 heads, head dim 128, 4096 tokens, fp16:", round(kv/1e6), "MB")
print("with 8 KV heads (GQA):", round(kv/4/1e6), "MB")

cache == recompute: True
K,V projections without cache: 21  with cache: 6
1024 tokens: score matrix per head in fp16 = 2.1 MB
8192 tokens: score matrix per head in fp16 = 134.2 MB
32768 tokens: score matrix per head in fp16 = 2147.5 MB
KV cache, 32 layers, 32 heads, head dim 128, 4096 tokens, fp16: 2147 MB
with 8 KV heads (GQA): 537 MB


## 11.5 A Mini-GPT in PyTorch (Not Run Here)

In [ ]:
# not run here
import torch, torch.nn as nn, torch.nn.functional as F

class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(self, x):
        T = x.size(1)
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), 1)  # True = blocked
        h = self.ln1(x)
        x = x + self.attn(h, h, h, attn_mask=mask, need_weights=False)[0]
        return x + self.ffn(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self, vocab, ctx=64, d=128, heads=4, layers=4):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab, d), nn.Embedding(ctx, d)
        self.blocks = nn.Sequential(*[Block(d, heads) for _ in range(layers)])
        self.ln_f, self.head = nn.LayerNorm(d), nn.Linear(d, vocab, bias=False)
    def forward(self, ids, targets=None):
        x = self.tok(ids) + self.pos(torch.arange(ids.size(1), device=ids.device))
        logits = self.head(self.ln_f(self.blocks(x)))
        loss = None if targets is None else F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        return logits, loss

## 11.6 Making Big Transformers Cheap: KV Cache, FlashAttention, GQA and MoE

In [ ]:
def kv(layers, kv_heads, dh, T, nbytes, batch=1):
    return 2*layers*kv_heads*dh*T*nbytes*batch
for name,h in [("MHA (32 KV heads)",32),("GQA (8 KV heads)",8),("MQA (1 KV head)",1)]:
    b=kv(32,h,128,4096,2)
    print(f"{name:20s} {b:>14,d} bytes  {b/2**30:.3f} GiB per sequence")

MHA (32 KV heads)     2,147,483,648 bytes  2.000 GiB per sequence
GQA (8 KV heads)        536,870,912 bytes  0.500 GiB per sequence
MQA (1 KV head)          67,108,864 bytes  0.062 GiB per sequence


In [ ]:
import numpy as np
s = np.array([1.0, 3.0, 2.0, 0.5, 4.0, 1.0])
v = np.array([[1.,0.],[0.,1.],[2.,2.],[1.,1.],[0.,3.],[3.,0.]])
full = np.exp(s-s.max()); full/=full.sum(); out_full = full@v
m, l, o = -np.inf, 0.0, np.zeros(2)
for b in range(0, 6, 3):
    sb, vb = s[b:b+3], v[b:b+3]
    m_new = max(m, sb.max())
    scale = np.exp(m-m_new) if np.isfinite(m) else 0.0
    p = np.exp(sb-m_new)
    l = l*scale + p.sum()
    o = o*scale + p@vb
    m = m_new
    print(f"block {b//3+1}: m={m:.1f} l={l:.4f} o={np.round(o,4)}")
print("online  :", np.round(o/l,4))
print("full    :", np.round(out_full,4))
print("equal   :", np.allclose(o/l, out_full))

block 1: m=3.0 l=1.5032 o=[0.8711 1.7358]
block 2: m=4.0 l=1.6330 o=[0.5    3.6687]
online  : [0.3062 2.2466]
full    : [0.3062 2.2466]
equal   : True


In [ ]:
import numpy as np
logits = np.array([2.0, 0.5, 1.5, -1.0])
top = np.argsort(-logits)[:2]
p = np.exp(logits[top]-logits[top].max()); p/=p.sum()
print("top-2 experts:", top, "weights:", np.round(p,4))
expert_out = {0: np.array([1.0, 2.0]), 2: np.array([3.0, 0.0])}
y = sum(w*expert_out[e] for e,w in zip(top,p))
print("mixed output:", np.round(y,4))

top-2 experts: [0 2] weights: [0.6225 0.3775]
mixed output: [1.7551 1.2449]


In [ ]:
d, h, kvh = 4096, 32, 8
dh = d//h
mha = 4*d*d
gqa = d*d + 2*d*(kvh*dh) + d*d
print("head dim", dh)
print("MHA attention params:", f"{mha:,}")
print("GQA attention params:", f"{gqa:,}", f"({gqa/mha:.3f} of MHA)")
dff = 14336
expert = 3*d*dff
print("one SwiGLU expert:", f"{expert:,}")
E_, k = 8, 2
router = d*E_
tot = E_*expert + router
act = k*expert + router
print("MoE FFN total:", f"{tot:,}", " active per token:", f"{act:,}", f"ratio {act/tot:.3f}")
dense = expert
print("FFN FLOPs per token (2 per weight): dense", f"{2*dense:,}", "MoE top-2", f"{2*act:,}")

head dim 128
MHA attention params: 67,108,864
GQA attention params: 41,943,040 (0.625 of MHA)
one SwiGLU expert: 176,160,768
MoE FFN total: 1,409,318,912  active per token: 352,354,304 ratio 0.250
FFN FLOPs per token (2 per weight): dense 352,321,536 MoE top-2 704,708,608


## 11.7 Fine-Tuning Big Models Cheaply: LoRA and QLoRA

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
d, r, alpha = 8, 2, 4
W = rng.normal(size=(d,d))
A = rng.normal(size=(r,d))*0.1
B = rng.normal(size=(d,r))*0.1
x = rng.normal(size=(3,d))
y_adapter = x@W.T + (alpha/r)*(x@A.T)@B.T
W_merged = W + (alpha/r)*B@A
y_merged = x@W_merged.T
print("max difference:", np.abs(y_adapter-y_merged).max() < 1e-12)
print("full params:", d*d, " LoRA params:", r*(d+d))
D, R = 4096, 8
print("4096x4096 full:", f"{D*D:,}", " rank 8 LoRA:", f"{R*2*D:,}", f"{R*2*D/(D*D)*100:.3f}%")
print("7B model, 32 layers, q and v adapted, r=8:", f"{32*2*R*2*D:,}")
print("fp16 base GiB", round(7e9*2/2**30,2), " 4-bit base GiB", round(7e9*0.5/2**30,2))

max difference: True
full params: 64  LoRA params: 32
4096x4096 full: 16,777,216  rank 8 LoRA: 65,536 0.391%
7B model, 32 layers, q and v adapted, r=8: 4,194,304
fp16 base GiB 13.04  4-bit base GiB 3.26


In [ ]:
from peft import LoraConfig, get_peft_model
config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.05)
model = get_peft_model(base_model, config)
model.print_trainable_parameters()   # prints trainable vs total counts

## Chapter recap

## Practice set

**1.** (easy) Scores for one query against three keys are `(2, 0, 0)`. Compute the softmax weights. **Answer:** `e² = 7.389`, `e⁰ = 1`, `1`. Sum `9.389`. Weights `(0.787, 0.1065, 0.1065)`, which add to 1.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch11.html